In [ ]:
import numpy as np
import numpy.typing as npt
from mag_den_implemenation import wrapper_mag_den
import channel_preprocess_utility as cu
import os

fixed_parameters = {}

name = '1_0_2'

# This is the initial guess value for scipy.optimize.minimize() 
fixed_parameters['robot_magnetization_density'] = 2.0e5

fixed_parameters['robot_length'] = 10.75
fixed_parameters['robot_width'] = 3.98
fixed_parameters['robot_thickness'] = 0.454
fixed_parameters['robot_density'] = 1.795 * 1e-3 # g/cm^3 to g/mm^3
fixed_parameters['robot_modulus'] = 2.2 # From 17/9/2025 Wang Hong's suggestion

target_dl = 0.05

fixed_parameters['OPTIMZATION_SCALE'] = 1e6

movie_names = ["9.7mT", "13.8mT"]
b_field_list = [0.0097, 0.0138]

# True will run scipy's least_squares(). False simply execute the run_crawling_sim()
fixed_parameters['should_optimize'] = True

static_mu_wall = 0.0
kinetic_mu_wall = 0.0
static_mu_wall_end = 0.0
kinetic_mu_wall_end = 0.0
static_mu_substrate = 0.0
kinetic_mu_substrate = 0.0

fixed_parameters['dist_threshold'] = fixed_parameters['robot_thickness'] / 2.0

n_elem = int(fixed_parameters['robot_length'] / target_dl)
if n_elem * target_dl < fixed_parameters['robot_length']:
    n_elem += 1
else:
    n_elem = n_elem
dl = fixed_parameters['robot_length'] / n_elem
dt = dl / 100.0  
fixed_parameters['robot_n_elem'] = n_elem

channel_length = fixed_parameters["robot_length"] * 2 + 2
channel_type = 'straight'

channel_list = []
channel_x_mid_list = []
channel_y_mid_list = []

robot_init_pos_list = []
robot_init_origin_list = []
robot_init_director_list = []

movie_names_list = []

width = fixed_parameters['robot_length'] * 2

# Prepare 5 same simulation environments
for i in range(len(b_field_list)):
    channel_x_mid, channel_y_mid, channel_polygon, left_bank, right_bank = cu.gen_channel_for_mag_den(
        channel_length,
        width,
        origin_offset=0.0
    )
    channel_list.append(channel_polygon)
    channel_x_mid_list.append(channel_x_mid)
    channel_y_mid_list.append(channel_y_mid)

    robot_x_init_pos = np.array([channel_length / 2, channel_length / 2])
    robot_y_init_pos = np.array([fixed_parameters["dist_threshold"] + 0.1 * width, fixed_parameters["dist_threshold"] + 0.1 * width + fixed_parameters["robot_length"]])

    robot_seg_midline = np.column_stack((robot_x_init_pos, robot_y_init_pos))
    robot_init_pos = cu.generate_fiber_in_segment(robot_seg_midline, L_fiber=fixed_parameters['robot_length'], offset_factor=0.0, N_fiber=fixed_parameters["robot_n_elem"])
    robot_init_pos_list.append(robot_init_pos)
    robot_init_director_list.append(cu.compute_directors_from_positions(robot_init_pos))

    robot_init_origin_list.append(robot_init_pos[:, 0])

    movie_names_list.append(f"{movie_names[i]}.mp4")

fixed_parameters['channel_list'] = channel_list

fixed_parameters['robot_init_pos_list'] = robot_init_pos_list
fixed_parameters["robot_init_origin_list"] = robot_init_origin_list
fixed_parameters["robot_init_director_list"] = robot_init_director_list
fixed_parameters["movie_names_list"] = movie_names_list

fixed_parameters['B_field_list'] = b_field_list

kinetic_mu_substrate_array = np.full((fixed_parameters['robot_n_elem'] + 1,), kinetic_mu_substrate)
static_mu_substrate_array = np.full((fixed_parameters['robot_n_elem'] + 1,), static_mu_substrate)
kinetic_mu_wall_array = np.full((fixed_parameters['robot_n_elem'] + 1,), kinetic_mu_wall)
static_mu_wall_array = np.full((fixed_parameters['robot_n_elem'] + 1,), static_mu_wall)
kinetic_mu_wall_array[:1] = kinetic_mu_wall_end
kinetic_mu_wall_array[-1:] = kinetic_mu_wall_end
static_mu_wall_array[:1] = static_mu_wall_end
static_mu_wall_array[-1:] = static_mu_wall_end

fixed_parameters["kinetic_mu_substrate_array"] = kinetic_mu_substrate_array
fixed_parameters["static_mu_substrate_array"] = static_mu_substrate_array
fixed_parameters["kinetic_mu_wall_array"] = kinetic_mu_wall_array
fixed_parameters["static_mu_wall_array"] = static_mu_wall_array
fixed_parameters['dt'] = dt
fixed_parameters['final_time'] = 40000
fixed_parameters['variation_level'] = 0.0

user_home_directory = os.path.expanduser('~')
txt_file_path = os.path.join(user_home_directory, 'Documents', 'GitHub', 'Bayesian_Optimization_for_Sheet_Robots', \
                             'mag_den', 'pdms_2', name, 'bend_angle.txt')

exp_bend_angle_list = []

with open(txt_file_path, 'r') as file:
    for line in file:
        bend_angle = float(line.strip())
        exp_bend_angle_list.append(bend_angle)
file.close()

exp_bend_angle_array: npt.NDArray[np.float64] = np.array(exp_bend_angle_list)
fixed_parameters['exp_bend_angle_array'] = exp_bend_angle_array

log_path = os.path.join(user_home_directory, 'Documents', 'GitHub', 'Bayesian_Optimization_for_Sheet_Robots', \
                             'mag_den', 'pdms_2', name, 'log.txt')
fixed_parameters['log_path'] = log_path

wrapper_mag_den(
    length=fixed_parameters["robot_length"],
    beam_width=fixed_parameters["robot_width"],
    beam_thickness=fixed_parameters["robot_thickness"],
    beam_density=fixed_parameters["robot_density"],
    modulus=fixed_parameters["robot_modulus"],
    magnetization_density=fixed_parameters["robot_magnetization_density"],
    channel_list=fixed_parameters["channel_list"],
    dist_threshold=fixed_parameters["dist_threshold"],
    robot_init_pos_list=fixed_parameters["robot_init_pos_list"],
    robot_init_origin_list=fixed_parameters["robot_init_origin_list"],
    robot_init_director_list=fixed_parameters["robot_init_director_list"],
    B_field_list=fixed_parameters["B_field_list"],
    kinetic_mu_substrate_array=fixed_parameters["kinetic_mu_substrate_array"],
    static_mu_substrate_array=fixed_parameters["static_mu_substrate_array"],
    kinetic_mu_wall_array=fixed_parameters["kinetic_mu_wall_array"],
    static_mu_wall_array=fixed_parameters["static_mu_wall_array"],
    exp_bend_angle_array=fixed_parameters['exp_bend_angle_array'],
    OPTIMIZATION_SCALE=fixed_parameters['OPTIMZATION_SCALE'],
    log_path=fixed_parameters['log_path'],
    dt=fixed_parameters["dt"],
    final_time=fixed_parameters["final_time"],
    movie_names_list=fixed_parameters["movie_names_list"],
    variation_level=fixed_parameters["variation_level"],
    should_optimize=fixed_parameters['should_optimize']
)

Optimization enabled.
------------------------------------------------

Iteration 0 relative_error (sim - exp) / exp:
[-0.04136673  0.01112367]
Simulated bend angles (degree, actual cal in radian):
[45.59922489 55.87026842]
Magnetization density:
200000.0 A/m

------------------------------------------------

Iteration 1 relative_error (sim - exp) / exp:
[-0.04136668  0.01112371]
Simulated bend angles (degree, actual cal in radian):
[45.59922707 55.87027054]
Magnetization density:
200000.0149011612 A/m

------------------------------------------------

Iteration 2 relative_error (sim - exp) / exp:
[-0.0226548   0.02658729]
Simulated bend angles (degree, actual cal in radian):
[46.48929351 56.72472047]
Magnetization density:
206143.0515057889 A/m

------------------------------------------------

Iteration 3 relative_error (sim - exp) / exp:
[-0.02265476  0.02658733]
Simulated bend angles (degree, actual cal in radian):
[46.48929563 56.72472251]
Magnetization density:
206143.0664069501 